# INSEA 2026 data challenge: starter notebook

Will this used car sell within 30 days? This notebook loads the competition data, looks at one ad, trains the kit's
baseline and writes a submission. It runs unchanged on Kaggle and on your laptop: the first cell finds the data.

- **On Kaggle**: **Copy & Edit** (top right). For a GPU: **Settings > Accelerator**. To download a pretrained model:
  **Settings > Internet: on**.
- **On your laptop**: run `just data` once, then `uv run jupyter lab` from the repository and open this notebook.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd

COMPETITION = "multimodal-predictive-ml-data-challenge"
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    INPUT = Path("/kaggle/input") / COMPETITION
    OUT = Path("/kaggle/working")
else:
    # the repository root is the first parent folder that holds the justfile, wherever Jupyter was started
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "justfile").exists())
    INPUT = ROOT / "data"
    OUT = ROOT / "submissions"
    assert (INPUT / "train.csv").exists(), f"no data in {INPUT}: run `just data` from the repository first"
OUT.mkdir(exist_ok=True)
print("on Kaggle" if ON_KAGGLE else "on a laptop", "| data:", INPUT, "| output:", OUT)
print(sorted(p.name for p in INPUT.iterdir()))

In [ ]:
# Kaggle unpacks the photos into images/images/, `just data` into data/images/
IMAGES = INPUT / "images"
if (IMAGES / "images").is_dir():
    IMAGES = IMAGES / "images"
print(IMAGES, len(list(IMAGES.glob("*.jpg"))), "photos")

In [ ]:
train = pd.read_csv(INPUT / "train.csv")
test = pd.read_csv(INPUT / "test.csv")
print(train.shape, test.shape, "train sale rate", round(train.sold_30d.mean(), 3))
train.head(3)

## One ad, three sources

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def photo_names(row):
    f = row["image_files"]
    return [] if pd.isna(f) or not f else f.split(";")

ad = train[train.image_files.notna()].iloc[0]
print(ad.drop(["title", "description", "image_files"]).to_string(), "\n")
print("TITLE:", ad.title, "\n\nDESCRIPTION:", ad.description)
names = photo_names(ad) if IMAGES.is_dir() else []
if not names: print("\n(no photos found: run `just data` again to get them)")
fig, axes = plt.subplots(1, max(len(names), 1), figsize=(3 * max(len(names), 1), 3))
for ax, name in zip(np.atleast_1d(axes), names):
    ax.imshow(Image.open(IMAGES / name)); ax.set_title(name, fontsize=8); ax.axis("off")
plt.show()

## Baseline: LDA on the table, complete rows only

Identical to `kit/train.py` in the repository: a linear discriminant analysis with default settings, on the numeric
columns and the one-hot encoded categorical columns, with no scaling and no tuning. It ignores the text and the photos,
drops every training ad with a missing value, and gives the train sale rate to the test ads with a missing value. Each of
these choices is a place to start.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

NUMERIC = ["reg_year", "km", "engine_l", "price_mad", "seats", "doors", "post_month"]
CATEGORICAL = ["make", "model", "gearbox", "fuel", "body_type", "color", "city", "seller_type"]
# we use only some of the features to speed up the training

base_rate = train["sold_30d"].mean()
cols = NUMERIC + CATEGORICAL
complete = train[cols].notna().all(axis=1)
print(f"train: {len(train):,} ads, {complete.sum():,} complete on the {len(cols)} columns used ({complete.mean():.1%})")
X, y = train.loc[complete, cols], train.loc[complete, "sold_30d"]

# numeric columns as they are, categorical columns one-hot encoded, then LDA: no scaling, no tuning
model = Pipeline([
    ("prep", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL)],
                               remainder="passthrough")),
    ("clf", LinearDiscriminantAnalysis()),
])
cv = -cross_val_score(model, X, y, cv=5, scoring="neg_log_loss")
print(f"5-fold log loss on the complete training rows: {cv.mean():.4f} (+/- {cv.std():.4f})")
model.fit(X, y)

test_complete = test[cols].notna().all(axis=1)
p = np.full(len(test), base_rate)
p[test_complete.values] = model.predict_proba(test.loc[test_complete, cols])[:, 1]
p = np.clip(p, 0.01, 0.99)
print(f"test: {len(test):,} ads, {(~test_complete).sum():,} with a missing value get the train sale rate {base_rate:.3f}")

## Write and check the submission

In [ ]:
sub = pd.DataFrame({"id": test.id, "sold_30d": p})
assert list(sub.columns) == ["id", "sold_30d"] and len(sub) == 19_919
assert set(sub.id) == set(test.id) and not sub.id.duplicated().any()
assert sub.sold_30d.notna().all() and sub.sold_30d.between(0, 1).all()
sub.to_csv(OUT / "submission.csv", index=False)
print(f"{OUT / 'submission.csv'}: {len(sub):,} rows, mean {sub.sold_30d.mean():.3f}, min {sub.sold_30d.min():.3f}, max {sub.sold_30d.max():.3f}")

To submit, one submission per day per team:

- **On Kaggle**: **Save Version** (top right, "Save & Run All"), then open the saved version, tab **Output**, and click
  **Submit to competition** next to `submission.csv`.
- **On your laptop**: from the repository, `just submit submissions/submission.csv "your message"`.

## GPU check on Kaggle

In [ ]:
try:
    import torch
    print("cuda available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU (on Kaggle: Settings > Accelerator)")
except ImportError:
    print("torch is not installed: `uv add torch` on your laptop (Kaggle has it already)")